In [2]:
import json

f = open('Datasets/wikipedia_val.json')
data = json.load(f)
resultados = []
for elem in data:
    if len(elem["Texto"]) > 500:
        resultados.append(elem)

In [5]:
import json
with open('Datasets/wiki_val_limpio.json', 'w', encoding='utf-8') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=4)

In [8]:
import random
res = random.sample(resultados, 150)

In [9]:
import json
with open('Datasets/wiki_val_limpio_def.json', 'w', encoding='utf-8') as f:
    json.dump(res, f, ensure_ascii=False, indent=4)

In [21]:
import pandas as pd

df = pd.read_csv("Datasets/ResultadoProposiciones_val_def.csv", encoding='latin-1', sep=';')

In [27]:
resultados = []
for index, row in df.iterrows():
    resultados.append({"Texto": row.Texto, "Proposiciones": list(filter(None, row.Respuesta.splitlines()))})

In [31]:
from datasets import Dataset

dataset = Dataset.from_list(resultados)

In [32]:
dataset.push_to_hub("ProposicionesValidacion_Gold-Standard")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/paascorb/ProposicionesValidacion_Gold-Standard/commit/90b74e234a61f1f5fb284c0ec87ceca15ac8d9bf', commit_message='Upload dataset', commit_description='', oid='90b74e234a61f1f5fb284c0ec87ceca15ac8d9bf', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/paascorb/ProposicionesValidacion_Gold-Standard', endpoint='https://huggingface.co', repo_type='dataset', repo_id='paascorb/ProposicionesValidacion_Gold-Standard'), pr_revision=None, pr_num=None)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("paascorb/ProposicionesValidacion_Gold-Standard")

## Gemma3

In [ ]:
import ollama

def obtener_respuesta(Texto):
    prompt=f"""Eres un chatbot encargado de extraer proposiciones atómicas de un texto. 
                Una proposición es atómica cuando es autocontenida: no necesita contexto u otra información para entenderse, por ejemplo: 'Morten Harket sufrió acoso escolar' es correcta mientras que 'Sufrió acoso escolar' no es autocontenida ya que no se sabe a quién hace referencia; es mínimal: no se puede descomponer en proposiciones más sencillas; y es coherente: tiene sentido, es decir, es una frase completa con sujeto y predicado, una palabra suelta o una sigla no es una proposición. 
                Solo debes listar las proposiciones atómicas separadas por puntos y muy importante un salto de línea (\\n) sin dar ninguna información extra; por ejemplo: París es una ciudad. París es la capital de Francia.
                Dado el siguiente texto de ejemplo: 'Morten Harket nació en Kongsberg siendo el segundo de 5 hijos. Su padre, Reidar, fue médico en un hospital y su madre Henny fue maestra de economía del hogar en una escuela. Morten fue un chico soñador y sufrió acoso escolar en la escuela. Se destacaba en las clases de religión y pensó ingresar en un seminario para ser ministro en la Iglesia Luterana. Entró a formar parte de A-ha el 14 de septiembre de 1982, por invitación de Paul Waaktaar-Savoy y Magne Furuholmen. Sin embargo antes de esto fue el vocalista de un grupo local llamado Soldier Blue. A-ha fue un grupo de gran éxito en los años 80, siendo su canción «Take on Me» la de mayor éxito, de su disco debut «Hunting High and low» vendiendo millones de discos, donde Morten Hartet se convirtió rápidamente en un rompe corazones por aquellas primeras imágenes del vídeo musical «Take on Me» en donde su aspecto de galán nórdico causó gran sensación entre el público femenino. Después de que A-ha se separara en 1994 por conflictos internos, Morten emprendió una carrera como solista cantante y compositor y participó en proyectos medioambientales. Más importante para él fue la lucha por liberar Timor Oriental de la ocupación indonesia, un objetivo que fue finalmente cumplido en 1999. En 1996, Morten es el encargado de presentar junto a Ingvild Bryn el Festival de Eurovisión que se celebró en Oslo, donde cantó Heaven's Not For Saints como introducción a la gala (no como representante de Noruega). Morten ha sacado tres álbumes como solista: «Poetenes Evangelium» (1993) y «Vogts Villa» (1996) en noruego y «Wild Seed» (1995) en inglés, este último fue un gran éxito en Noruega, vendiendo más de 160,000 copias. Otros proyectos de Morten son: la banda sonora de la película «Kamilla og Tyven» (1987), en la que además interpretó un personaje, y «Coneheads» (1993). Otras bandas sonoras incluyen el tema central del musical «Sophie's World», «A Jester In Out Town» y la canción «Jungle Of Beliefs» del disco «Cultures Span The World». En el año 2007 presenta «Letter from Egypt» su regreso como solista y dando una pausa a su participación con el trío noruego A-ha. El 14 de noviembre de 2007 se difundió su primer sencillo «Movies» en la radio en Noruega, colocándose rápidamente en los primeros lugares. En diciembre de 2007 tuvo una presentación en el Premio Nobel de la Paz.'
                Serían proposiciones atómicas correctas: Morten Harket nació en Kongsberg. Morten Harket es el segundo de 5 hijos. El padre de Morten Harket se llama Reidar. Reidar Harket fue médico de un hospital.
                Serían proposciones átomicas incorrectas: Él nació en Kongsberg. (No autocontenida); Su madre fue maestra de economía del hogar. (No autocontenida); Morten Harket fue un chico soñador y sufrió acoso escolar. (No minimal, se puede separar en: Morten Harket fue un chico soñador. Morten Harket sufrió acoso escolar.); Morten Harket fue astronauta. (Incorrecta, ya que esa infromación no está en el texto);
                Por último, es importante que si no logras extraer proposiciones de ninguna forma pongas: 'No se pueden extaer proposiciones.'."\n
                Dado el siguiente texto: {Texto}\n
                Lista las proposiciones atómicas:
           """
    return ollama.generate(model='gemmaCustomP', prompt=prompt)["response"]

In [ ]:
import time

resultados_gemma = []
cont = 1
tam = len(dataset["train"])
for elem in dataset["train"]:
    print(f"Fase {cont} de {tam}", end='\r')
    texto = elem["Texto"]
    start = time.time()
    respuesta = obtener_respuesta(texto)
    end = time.time()
    resultados_gemma.append({"Texto": texto, "Proposiciones": list(filter(None, respuesta.splitlines())), "Tiempo": end - start})
    cont += 1

In [ ]:
import pandas
df_gemma = pandas.DataFrame(data=resultados_gemma)
df_gemma.to_csv("Datasets/evaluacion_proposiciones_gemma3.csv", index=False)

## mT5-large

In [ ]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained("paascorb/ProposicionadorES-mT5-large", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("paascorb/ProposicionadorES-mT5-large")

In [ ]:
def obtener_respuesta_mt5(texto):
    input_ids = tokenizer(texto, return_tensors="pt").input_ids.to("cuda")
    
    outputs = model.generate(input_ids, max_length=2048)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import time

resultados_mT5 = []
cont = 1
tam = len(dataset["train"])
for elem in dataset["train"]:
    print(f"Fase {cont} de {tam}", end='\r')
    texto = elem["Texto"]
    start = time.time()
    respuesta = obtener_respuesta_mt5(texto)
    end = time.time()
    resultados_mT5.append({"Texto": texto, "Proposiciones": list(filter(None, respuesta.splitlines())), "Tiempo": end - start})
    cont += 1

In [ ]:
df_mt5 = pandas.DataFrame(data=resultados_mT5)
df_mt5.to_csv("Datasets/evaluacion_proposiciones_byt5-large.csv", index=False)

## LiquidAI LFM2-1.2B

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

# Load model and tokenizer
model_id = "paascorb/ProposicionadorES-LFM2-1.2B"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="bfloat16",
)
tokenizer = AutoTokenizer.from_pretrained("LiquidAI/LFM2-1.2B")

In [ ]:
def obtener_respuesta_LFM2(texto):
    
    prompt = texto
    input_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
        tokenize=True,
    ).to(model.device)
    
    output = model.generate(
        input_ids,
        do_sample=True,
        temperature=0.3,
        min_p=0.15,
        repetition_penalty=1.05,
        max_new_tokens=2048,
    )
    
    a = tokenizer.decode(output[0], skip_special_tokens=False).split("<|im_start|>assistant")[1].strip("<|im_end|>")
    return a

In [ ]:
import time

resultados_LFM2 = []
cont = 1
tam = len(dataset["train"])
for elem in dataset["train"]:
    print(f"Fase {cont} de {tam}", end='\r')
    texto = elem["Texto"]
    start = time.time()
    respuesta = obtener_respuesta_LFM2(texto)
    end = time.time()
    resultados_LFM2.append({"Texto": texto, "Proposiciones": list(filter(None, respuesta.splitlines())), "Tiempo": end - start})
    cont += 1

In [ ]:
df_LFM2 = pandas.DataFrame(data=resultados_LFM2)
df_LFM2.to_csv("Datasets/evaluacion_proposiciones_LFM2_1,2B.csv", index=False)

## EuroLLM Utter

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "paascorb/ProposicionadorES-EuroLLM-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id).to('cuda')

In [ ]:
def obtener_respuesta_EuroLLM(texto):
    text = f'<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{texto}<|im_end|>\n<|im_start|>assistant\n'

    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, temperature=0.3,
                            min_p=0.15,
                            repetition_penalty=1.05,
                            max_new_tokens=2048)
    return tokenizer.decode(outputs[0], skip_special_tokens=False).split("<|im_start|> assistant")[1].strip("<|im_end|>")

In [ ]:
import time

resultados_Euro = []
cont = 1
tam = len(dataset["train"])
for elem in dataset["train"]:
    print(f"Fase {cont} de {tam}", end='\r')
    texto = elem["Texto"]
    start = time.time()
    respuesta = obtener_respuesta_EuroLLM(texto)
    end = time.time()
    resultados_Euro.append({"Texto": texto, "Proposiciones": list(filter(None, respuesta.splitlines())), "Tiempo": end - start})
    cont += 1

In [ ]:
df_Euro = pandas.DataFrame(data=resultados_Euro)
df_Euro.to_csv("Datasets/evaluacion_proposiciones_EuroLLM-1,7B.csv", index=False)